# Alquiler - 02 Preparacion

Preparacion inicial de datos para un futuro modelo de estimacion de alquiler mensual.

## 0. Configuracion del entorno

Ejecutar esta celda solo la primera vez que se configura el entorno o cuando se modifiquen las dependencias en `requirements.txt`.

In [ ]:
%pip install -r ../requirements.txt

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path('..') / 'Data'

pd.set_option('display.max_columns', 100)

## Carga de datos

In [ ]:
rentals = pd.read_csv(DATA_DIR / 'rentals.csv')
district_prices = pd.read_csv(DATA_DIR / 'district_prices_monthly.csv')
metro = pd.read_csv(DATA_DIR / 'metro_stations.csv')

print(rentals.shape, district_prices.shape, metro.shape)

## Funciones auxiliares

In [ ]:
def clean_column_names(df):
    out = df.copy()
    out.columns = (
        out.columns
        .str.strip()
        .str.lower()
        .str.replace(r'[^0-9a-zA-Z]+', '_', regex=True)
        .str.strip('_')
    )
    return out


def drop_full_duplicates(df):
    return df.drop_duplicates().reset_index(drop=True)


def coerce_numeric_like_columns(df, min_success=0.85):
    out = df.copy()
    object_cols = out.select_dtypes(include='object').columns

    for col in object_cols:
        series = out[col]
        if series.notna().mean() == 0:
            continue
        cleaned = (
            series.astype(str)
            .str.replace(',', '.', regex=False)
            .str.replace(r'[^0-9.-]+', '', regex=True)
        )
        converted = pd.to_numeric(cleaned, errors='coerce')
        if converted.notna().mean() >= min_success:
            out[col] = converted

    return out


def missing_report(df):
    return (
        pd.DataFrame({
            'tipo': df.dtypes.astype(str),
            'nulos': df.isna().sum(),
            'pct_nulos': df.isna().mean().round(4),
            'unicos': df.nunique(dropna=True),
        })
        .sort_values('pct_nulos', ascending=False)
    )

## Limpieza base

In [ ]:
rentals_clean = coerce_numeric_like_columns(drop_full_duplicates(clean_column_names(rentals)))
district_prices_clean = coerce_numeric_like_columns(drop_full_duplicates(clean_column_names(district_prices)))
metro_clean = coerce_numeric_like_columns(drop_full_duplicates(clean_column_names(metro)))

rent = rentals_clean.copy()

print(rent.shape)
display(rent.head())

In [ ]:
display(missing_report(rent).head(30))

## Variables derivadas candidatas

In [ ]:
rent_cols = [col for col in rent.columns if any(token in col for token in ['rent', 'rental', 'price', 'cost', 'value'])]
area_cols = [col for col in rent.columns if any(token in col for token in ['area', 'sqm', 'sq_m', 'm2'])]

print('Columnas candidatas de alquiler:', rent_cols)
print('Columnas candidatas de superficie:', area_cols)

if rent_cols and area_cols:
    rent_col = rent_cols[0]
    area_col = area_cols[0]
    rent['rent_per_area_candidate'] = rent[rent_col] / rent[area_col].replace(0, np.nan)
    display(rent[[rent_col, area_col, 'rent_per_area_candidate']].describe().T)

In [ ]:
date_like_cols = [col for col in rent.columns if any(token in col for token in ['date', 'month', 'year'])]

for col in date_like_cols:
    parsed = pd.to_datetime(rent[col], errors='coerce')
    if parsed.notna().mean() >= 0.5:
        rent[col] = parsed
        rent[f'{col}_year'] = rent[col].dt.year
        rent[f'{col}_month'] = rent[col].dt.month

display(rent.head())

## Dataset preparado en memoria

La variable `rent` queda como base inicial para continuar con seleccion de variables, imputacion, codificacion y modelado.